# Topic 16 — Feature Engineering
### Theory → tiny example → sklearn tools → experiment.

**Feature engineering** = turning raw data into the numeric inputs a model can actually use, and
shaping those inputs to make patterns easier for the model to find. Often the single highest-leverage
step in a real ML project — better features usually beat a fancier algorithm.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold

rng = np.random.default_rng(0)

## 1. Missing-value handling

Options: drop rows/columns (Topic 2's `dropna`), or **impute** (fill in) missing values using a
strategy like mean, median, most-frequent, or a constant.

In [ ]:
df = pd.DataFrame({
    "age": [25, np.nan, 40, 35, np.nan],
    "income": [40000, 42000, np.nan, 51000, 39000],
})
print("before:\n", df)

imputer = SimpleImputer(strategy="median")
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
print("\nafter median imputation:\n", df_imputed)
# IMPORTANT: fit the imputer on TRAIN data only, then .transform() test data -- same leakage rule as scaling.

## 2. Categorical encoding

Models need numbers, not text categories. Two common approaches:

- **One-hot encoding**: each category becomes its own 0/1 column. No false ordering implied.
  Best for *nominal* categories (no natural order) — e.g. `platform: insta/twitter/youtube`.
- **Ordinal encoding**: each category becomes a single integer. Implies an ORDER.
  Only appropriate for *ordinal* categories — e.g. `severity: low/medium/high`.

In [ ]:
platform_df = pd.DataFrame({"platform": ["insta", "twitter", "youtube", "insta", "twitter"]})

ohe = OneHotEncoder(sparse_output=False)
platform_onehot = ohe.fit_transform(platform_df[["platform"]])
print("one-hot columns:", ohe.get_feature_names_out())
print(platform_onehot)

# pandas' built-in shortcut for the same thing
print("\npd.get_dummies:\n", pd.get_dummies(platform_df["platform"]))

In [ ]:
severity_df = pd.DataFrame({"severity": ["low", "high", "medium", "low", "high"]})

# Must tell OrdinalEncoder the correct order -- it won't guess "low < medium < high" on its own
oe = OrdinalEncoder(categories=[["low", "medium", "high"]])
severity_encoded = oe.fit_transform(severity_df[["severity"]])
print(np.hstack([severity_df.values, severity_encoded]))
# low -> 0, medium -> 1, high -> 2. Using one-hot here would THROW AWAY the meaningful order.

## 3. Scaling & normalization

Covered fully in Topic 15 — `StandardScaler`, `MinMaxScaler`. Included here because in practice
it's one step inside the larger feature engineering process, usually applied to numeric columns
right before modeling.

In [ ]:
numeric_df = pd.DataFrame({"age": [25, 40, 35, 60], "income": [40000, 65000, 51000, 90000]})
scaled = StandardScaler().fit_transform(numeric_df)
print(scaled)

## 4. Outlier handling

Two common strategies: **capping** (clip extreme values to a boundary) or **removing** rows
entirely. Use domain judgment — an "outlier" might be a data error, or a genuinely rare but valid case.

In [ ]:
values = np.array([20, 22, 21, 23, 19, 500, 24])   # 500 is likely an error, or a genuine extreme case

q1, q3 = np.percentile(values, [25, 75])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
print("IQR bounds:", lower_bound, upper_bound)

# Capping: clip values to the bounds instead of deleting them
capped = np.clip(values, lower_bound, upper_bound)
print("original:", values)
print("capped:  ", capped)

# Removing: drop rows outside the bounds entirely
removed = values[(values >= lower_bound) & (values <= upper_bound)]
print("removed: ", removed)

## 5. Feature selection

Not all features help — some are noise, some are redundant. Selecting a good subset can reduce
overfitting and speed up training.

- **VarianceThreshold**: drop features that barely vary (carry almost no information).
- **SelectKBest**: keep the `k` features most statistically related to the target.

In [ ]:
X = np.array([
    [1, 100, 0.001],
    [2, 200, 0.002],
    [3, 150, 0.001],
    [4, 300, 0.003],
])   # column 2 barely changes -- almost no variance

vt = VarianceThreshold(threshold=0.01)
X_reduced = vt.fit_transform(X)
print("kept feature indices:", vt.get_support(indices=True))
print("reduced X:\n", X_reduced)

# SelectKBest: pick the 2 features most correlated with the target
X_clf = rng.normal(0, 1, size=(100, 5))
y_clf = (X_clf[:, 0] + X_clf[:, 2] * 2 + rng.normal(0, 0.1, 100) > 0).astype(int)   # only cols 0,2 matter

selector = SelectKBest(score_func=f_classif, k=2)
X_selected = selector.fit_transform(X_clf, y_clf)
print("\nselected feature indices:", selector.get_support(indices=True))
# Should correctly identify columns 0 and 2 as the informative ones.

## 6. Feature extraction & transformation

**Extraction**: deriving new features from raw/unstructured data (e.g. `text_length` and
`num_exclamations` extracted from raw text — a preview of Topic 21's NLP preprocessing).
**Transformation**: reshaping an existing feature's distribution (e.g. log-transform to fix skew).

In [ ]:
raw_texts = pd.Series([
    "you are stupid!!!", "have a nice day", "i hate you so much!!",
])

# Feature extraction: turn raw text into numeric features
extracted = pd.DataFrame({
    "text_length": raw_texts.str.len(),
    "num_exclamations": raw_texts.str.count("!"),
    "num_words": raw_texts.str.split().str.len(),
})
print(extracted)

# Feature transformation: log-transform a right-skewed feature (recall Topic 3's skewed histogram)
skewed = rng.gamma(2, 20, 1000)
log_transformed = np.log1p(skewed)   # log1p = log(1+x), safely handles zero values

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(skewed, bins=30); axes[0].set_title("before log transform (skewed)")
axes[1].hist(log_transformed, bins=30); axes[1].set_title("after log transform (more symmetric)")
plt.tight_layout()
plt.show()

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Build a small DataFrame with a missing categorical column, impute it with strategy="most_frequent".
# 2. One-hot encode a column with 5 categories and confirm you get 5 new binary columns.
# 3. Use SelectKBest with k=3 on a dataset you build with 6 features (only 3 informative) --
#    check whether it correctly picks the informative ones.
# 4. Extract 3 new numeric features from this list of texts: has_url, has_mention, num_capital_words
#    texts = ["check this out http://x.com", "@john you are wrong", "HELLO EVERYONE"]

---
### Next up: **Topic 17 — Pipelines** (chaining all these preprocessing steps + a model, safely).

Say "next" when you're ready.